In [ ]:

class PIController:
    #PI controller with output limits and conditional anti-windup

    def __init__(
        self, Kp, Ki, dt,
        u_min=None, u_max=None,
        anti_windup=True
    ):
        self.Kp = Kp
        self.Ki = Ki
        self.dt = dt
        self.u_min = u_min
        self.u_max = u_max
        self.anti_windup = anti_windup
        self.integral = 0.0

    def _clamp(self, u):
        if self.u_max is not None:
            u = min(u, self.u_max)
        if self.u_min is not None:
            u = max(u, self.u_min)
        return u

    def update(self, setpoint, measurement): ##vvvv important
        error = setpoint - measurement

        candidate = (
            self.integral
            + self.Ki * error * self.dt
        )

        u_unsat = self.Kp * error + candidate
        u = self._clamp(u_unsat)

        if self.anti_windup:
            winding_up = (
                (u_unsat > u and error > 0)
                or (u_unsat < u and error < 0)
            )

            if not winding_up:
                self.integral = candidate
        else:
            self.integral = candidate

        return u

    def reset(self):
        self.integral = 0.0


PROPERTY EFFECT
- Steady-state error (constant setpoint/disturbance) : Eliminated, regardless of plant parameters


- Speed of error removal Increases with Ki : Overshoot / oscillation Increases with Ki (reduces damping)

- System order Raised by one; can destabilize a loop that P alone kept stable
- Saturation Causes windup; requires anti-windup
Ki changing at runtime Use the “integrate Ki·e” form to avoid jumps
